# SCOUT DE JOVENES PROMESAS DE FUTBOL
## PASO 1 - EXTRACCION, EXPLORACION Y VISUALIZACION

**Objetivo del proyecto:**  
Construir un sistema de scouting basado en datos históricos que permita identificar 
jugadores jóvenes con mayor potencial de convertirse en estrellas, usando el 
**valor de mercado futuro** como proxy de ese potencial.

**Fuente de datos:** Transfermarkt (vía Kaggle)  
**Archivos disponibles:** 11 CSV con información de jugadores, rendimiento, 
lesiones, transferencias y valoraciones de mercado.

**Fecha de referencia para cálculo de edad:** 2025-01-01

Se hacen los imports, se configura paleta de colores y se definen las funciones necesarias

In [6]:
# IMPORTS
# ──────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")

# CONFIGURACIÓN
# ──────────────────────────────────────────────────────────────
DATA_DIR   = Path("../data")
OUTPUT_DIR = Path("../outputs/paso1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REF_DATE = pd.Timestamp("2025-01-01")  # Fecha de referencia para calcular edad

PALETTE = {
    "primary":   "#1D9E75",
    "secondary": "#7F77DD",
    "accent":    "#EF9F27",
    "danger":    "#D85A30",
    "neutral":   "#888780",
    "light":     "#F4F3EE",
    "dark":      "#2C2C2A",
}
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.facecolor": PALETTE["light"],
    "axes.facecolor":   PALETTE["light"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size": 11,
})


def save_fig(fig, name: str):
    path = OUTPUT_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=PALETTE["light"])
    plt.close(fig)
    print(f"  ✓ {name}.png")


def section(title: str):
    print(f"\n{'═'*62}")
    print(f"  {title}")
    print(f"{'═'*62}")

## 1. Carga de archivos

Se cargan los 8 archivos relevantes para el modelo. Los tres restantes 
(`team_children`, `team_competitions_seasons`, `player_teammates_played_with`) 
están disponibles pero no se usan en esta fase — podrían incorporarse en el 
Paso 2 como features adicionales (por ejemplo, calidad de compañeros de equipo 
como señal de nivel competitivo).

In [7]:
# 1. CARGA DE ARCHIVOS
# ──────────────────────────────────────────────────────────────

section("1. CARGA DE ARCHIVOS")

profiles  = pd.read_csv(DATA_DIR / "player_profiles.csv",                low_memory=False)
mv_hist   = pd.read_csv(DATA_DIR / "player_market_value.csv")
mv_latest = pd.read_csv(DATA_DIR / "player_latest_market_value.csv")
perfs     = pd.read_csv(DATA_DIR / "player_performances.csv",            low_memory=False)
injuries  = pd.read_csv(DATA_DIR / "player_injuries.csv")
national  = pd.read_csv(DATA_DIR / "player_national_performances.csv")
transfers = pd.read_csv(DATA_DIR / "transfer_history.csv",               low_memory=False)
team_det  = pd.read_csv(DATA_DIR / "team_details.csv")

for name, df in [
    ("player_profiles",                profiles),
    ("player_market_value",            mv_hist),
    ("player_latest_market_value",     mv_latest),
    ("player_performances",            perfs),
    ("player_injuries",                injuries),
    ("player_national_performances",   national),
    ("transfer_history",               transfers),
    ("team_details",                   team_det),
]:
    print(f"  {name:<35} {len(df):>10,} filas  ×  {df.shape[1]} cols")


══════════════════════════════════════════════════════════════
  1. CARGA DE ARCHIVOS
══════════════════════════════════════════════════════════════
  player_profiles                         92,671 filas  ×  34 cols
  player_market_value                    901,429 filas  ×  3 cols
  player_latest_market_value              69,441 filas  ×  3 cols
  player_performances                  1,878,719 filas  ×  20 cols
  player_injuries                        143,195 filas  ×  7 cols
  player_national_performances            92,701 filas  ×  9 cols
  transfer_history                     1,101,440 filas  ×  10 cols
  team_details                             2,175 filas  ×  12 cols


## 2. Exploración básica

Antes de construir nada, inspeccionamos la estructura de cada archivo:
tipos de datos, valores nulos, rangos y distribuciones generales.

El objetivo de esta sección es responder:
- ¿Qué tan completos están los datos?
- ¿Hay columnas con demasiados nulos para ser útiles?
- ¿Los rangos de valores tienen sentido (edades, valores en euros, minutos jugados)?
- ¿Hay duplicados o inconsistencias evidentes?

In [8]:
# 2. EXPLORACIÓN BÁSICA
# ──────────────────────────────────────────────────────────────

section("2. EXPLORACIÓN BÁSICA")

# descripcion de las columnas con la informacion de los jugadores
print("\n  player_profiles — columnas:")
print(f"  {list(profiles.columns)}\n")

# Calcular edad
profiles["date_of_birth"] = pd.to_datetime(profiles["date_of_birth"], errors="coerce")
profiles["age"] = ((REF_DATE - profiles["date_of_birth"]).dt.days / 365.25).round(1)

print("  Distribución de edades (todos los jugadores):")
print(profiles["age"].describe().round(1).to_string())

print("\n  Posiciones (main_position):")
print(profiles["main_position"].value_counts().to_string())

print("\n  Rango histórico de valoraciones:")
mv_hist["date_unix"] = pd.to_datetime(mv_hist["date_unix"], errors="coerce")
print(f"  Desde: {mv_hist['date_unix'].min().date()}  Hasta: {mv_hist['date_unix'].max().date()}")
print(f"  Valor máximo registrado: {mv_hist['value'].max()/1e6:.0f}M€")
print(f"  Valor mediano: {mv_hist['value'].median()/1e6:.2f}M€")


══════════════════════════════════════════════════════════════
  2. EXPLORACIÓN BÁSICA
══════════════════════════════════════════════════════════════

  player_profiles — columnas:
  ['player_id', 'player_slug', 'player_name', 'player_image_url', 'name_in_home_country', 'date_of_birth', 'place_of_birth', 'country_of_birth', 'height', 'citizenship', 'is_eu', 'position', 'main_position', 'foot', 'current_club_id', 'current_club_name', 'joined', 'contract_expires', 'outfitter', 'social_media_url', 'player_agent_id', 'player_agent_name', 'contract_option', 'date_of_last_contract_extension', 'on_loan_from_club_id', 'on_loan_from_club_name', 'contract_there_expires', 'second_club_url', 'second_club_name', 'third_club_url', 'third_club_name', 'fourth_club_url', 'fourth_club_name', 'date_of_death']

  Distribución de edades (todos los jugadores):
count    91665.0
mean        30.3
std          9.2
min         14.2
25%         22.8
50%         28.6
75%         36.5
max         72.9

  Posicione

### Hallazgos de la exploración inicial

- **92,671 jugadores** en total en `player_profiles`, con edades entre 14 y 73 años.
  El dataset incluye jugadores históricos y retirados, por eso el promedio de edad
  es 30.3 años — mucho más alto de lo esperado para un análisis de promesas.

- **901,429 registros** de valoraciones históricas desde 2003 hasta septiembre 2025,
  cubriendo 69,441 jugadores únicos. El valor mediano histórico es de apenas 0.30M€,
  lo que refleja que la gran mayoría de jugadores profesionales tienen valoraciones modestas.

- **1,878,719 registros** de rendimiento en `player_performances` abarcando 102 temporadas
  distintas — esto incluye desde ligas amateur hasta Champions League.

- `player_profiles` tiene 34 columnas, muchas de ellas no relevantes para el modelo
  (URLs de imágenes, redes sociales, agentes). Se seleccionaron solo 15 columnas útiles.


## 3. Procesamiento de cada fuente

Cada archivo se transforma individualmente antes del merge final.
El criterio general es **agregar a nivel de jugador** (una fila por jugador)
para poder hacer el join con el perfil base.

Las transformaciones principales son:
- **Valor de mercado histórico:** calcular valor máximo, medio, crecimiento absoluto 
  y porcentual, y número de valoraciones registradas.
- **Rendimiento:** sumar goles, asistencias y minutos de toda la carrera. 
  Calcular eficiencia (minutos por gol) y amplitud competitiva (nº de ligas distintas).
- **Lesiones:** contar número total de lesiones y días de baja acumulados.
- **Selección nacional:** sumar partidos y goles internacionales, 
  identificar si es convocado actualmente.
- **Transferencias:** contar movimientos de tipo *Transfer* (excluyendo préstamos y 
  retornos) y capturar el fee máximo pagado por el jugador.

In [9]:
# 3. PROCESAMIENTO DE CADA FUENTE
# ──────────────────────────────────────────────────────────────

section("3. PROCESAMIENTO DE FUENTES")

# ── 3a. Último valor de mercado
# se obtiene el ultimo valor de mercado y s filtra eliminando a los jugadores que no tienen valor de mercado
mv_latest.columns = ["player_id", "latest_date", "latest_value"]
mv_latest["latest_date"] = pd.to_datetime(mv_latest["latest_date"], errors="coerce")
mv_latest = mv_latest[mv_latest["latest_value"] > 0]
print(f"  latest_market_value: {len(mv_latest):,} jugadores con valor > 0")

# ── 3b. Histórico: valor máximo, crecimiento, n_valoraciones
# se obtiene el maximo valor de cada jugador y se crean medidas como: valor promedio, primer y ultimo valor (valor absoluto de crecimiento y porcentaje del mismo)
mv_hist = mv_hist[mv_hist["value"] > 0]
mv_agg = (
    mv_hist.sort_values("date_unix")
    .groupby("player_id")
    .agg(
        value_max         = ("value", "max"),
        value_mean        = ("value", "mean"),
        n_valuations      = ("value", "count"),
        value_first       = ("value", "first"),
        value_last        = ("value", "last"),
        date_first        = ("date_unix", "first"),
        date_last         = ("date_unix", "last"),
    )
    .reset_index()
)
mv_agg["value_growth_abs"] = mv_agg["value_last"] - mv_agg["value_first"]
mv_agg["value_growth_pct"] = (
    (mv_agg["value_last"] / mv_agg["value_first"].replace(0, np.nan)) - 1
).round(3)
print(f"  mv_hist agregado:    {len(mv_agg):,} jugadores")

# ── 3c. Estadísticas de carrera
# se obtienen las estadisticas de carrera (goles, asistencias, minutos jugados, tarjetas) y se claculan metricas clave para evaluar el rendimiento del jugador como: contribuciones de gol (goles + asistencias) y minutos por cada gol anotado (minutos jugados / goles anotados)
perfs["goals"]          = pd.to_numeric(perfs["goals"],           errors="coerce")
perfs["minutes_played"] = pd.to_numeric(perfs["minutes_played"],  errors="coerce")
perfs["assists"]        = pd.to_numeric(perfs["assists"],          errors="coerce")

career_stats = (
    perfs.groupby("player_id")
    .agg(
        total_goals      = ("goals",           "sum"),
        total_assists    = ("assists",          "sum"),
        total_minutes    = ("minutes_played",   "sum"),
        total_yellow     = ("yellow_cards",     "sum"),
        total_red        = ("direct_red_cards", "sum"),
        n_competitions   = ("competition_id",   "nunique"),
        n_seasons_active = ("season_name",      "nunique"),
    )
    .reset_index()
)
career_stats["goal_contrib"]  = career_stats["total_goals"] + career_stats["total_assists"]
career_stats["mins_per_goal"] = (
    career_stats["total_minutes"] / career_stats["total_goals"].replace(0, np.nan)
).round(1)
print(f"  career_stats:        {len(career_stats):,} jugadores")

# ── 3d. Lesiones
# se obtinen datos relevantes de lesiones, razon, dias y juegos perdidos
inj_agg = (
    injuries.groupby("player_id")
    .agg(
        n_injuries         = ("injury_reason", "count"),
        total_days_missed  = ("days_missed",   "sum"),
        total_games_missed = ("games_missed",  "sum"),
    )
    .reset_index()
)
print(f"  inj_agg:             {len(inj_agg):,} jugadores con historial de lesiones")

# ── 3e. Selección nacional
# se obtienen datos de seleccion nacional como: partidos jugados, goles, si es jugador de seleccion actualmente
nat_agg = (
    national.groupby("player_id")
    .agg(
        intl_matches        = ("matches",      "sum"),
        intl_goals          = ("goals",        "sum"),
        n_nat_teams         = ("team_id",      "nunique"),
        is_current_national = (
            "career_state",
            lambda x: int("CURRENT_NATIONAL_PLAYER" in x.values)
        ),
    )
    .reset_index()
)
print(f"  nat_agg:             {len(nat_agg):,} jugadores con participación internacional")

# ── 3f. Transferencias (solo tipo Transfer)
# datos de tranferencia, se filtra unicamente dejando solo aquellos que fueron realmente tranferidos y se calcula el valor maximo y la sumatoria
transfers["transfer_fee"]      = pd.to_numeric(transfers["transfer_fee"],      errors="coerce").fillna(0)
transfers["value_at_transfer"] = pd.to_numeric(transfers["value_at_transfer"], errors="coerce").fillna(0)

transfer_agg = (
    transfers[transfers["transfer_type"] == "Transfer"]
    .groupby("player_id")
    .agg(
        n_transfers = ("transfer_type", "count"),
        max_fee     = ("transfer_fee",  "max"),
        total_fee   = ("transfer_fee",  "sum"),
    )
    .reset_index()
)
print(f"  transfer_agg:        {len(transfer_agg):,} jugadores con al menos 1 transferencia")


══════════════════════════════════════════════════════════════
  3. PROCESAMIENTO DE FUENTES
══════════════════════════════════════════════════════════════
  latest_market_value: 33,590 jugadores con valor > 0
  mv_hist agregado:    69,404 jugadores
  career_stats:        88,375 jugadores
  inj_agg:             34,561 jugadores con historial de lesiones
  nat_agg:             36,339 jugadores con participación internacional
  transfer_agg:        117,674 jugadores con al menos 1 transferencia


### Hallazgos del procesamiento

- **Lesiones:** solo 34,561 jugadores tienen historial de lesiones registrado.
  Para el resto se asigna 0, lo cual es correcto — ausencia de registro se interpreta
  como ausencia de lesiones documentadas, no como dato faltante.

- **Selección nacional:** 2,936 jugadores tienen estado `CURRENT_NATIONAL_PLAYER`.
  Esta es una señal fuerte de nivel — los convocados actualmente tienden a tener
  valoraciones significativamente más altas.

- **Transferencias:** el fee mediano es 0€ porque la mayoría de movimientos son
  libres o préstamos. El máximo registrado es 222M€. Se filtraron solo las
  transferencias de tipo *Transfer* para evitar ruido de préstamos y retornos.

- **Rendimiento:** los minutos jugados tienen bastantes nulos en algunas temporadas
  antiguas del dataset — esto afecta el cálculo de `mins_per_goal` (solo 71.8% completo).
  Se tratará con imputación.

## 4. Construcción del dataset maestro

Se aplica el filtro central del proyecto: **jugadores ≤ 26 años con club activo**
(excluyendo "Retired" y "Without Club").

Este corte de edad se justifica porque:
- El objetivo es identificar *promesas*, no jugadores en el pico o declive de su carrera.
- El valor de mercado crece de forma más predecible y acelerada antes de los 26 años.
- Los datos de rendimiento son suficientemente ricos para jugadores que ya tienen
  al menos una temporada profesional registrada.

Después del filtro se encadenan 7 merges sucesivos usando `player_id` como llave,
todos con `how='left'` para conservar todos los jugadores del filtro base aunque
no tengan registro en alguna fuente secundaria.

In [11]:
# 4. CONSTRUCCIÓN DEL DATASET MAESTRO
# ──────────────────────────────────────────────────────────────

section("4. CONSTRUCCIÓN DEL DATASET MAESTRO")

# Filtro base: jóvenes promesas ≤ 26 años con club activo, se excluyen jugadores sin equipo y retirados
young = profiles[
    (profiles["age"] >= 14) &
    (profiles["age"] <= 26) &
    (profiles["current_club_name"].notna()) &
    (~profiles["current_club_name"].isin(["Retired", "Without Club"]))
].copy()
print(f"  Base — jugadores ≤26 años con club activo: {len(young):,}")

keep_profile = [
    "player_id", "player_name", "date_of_birth", "age",
    "country_of_birth", "citizenship", "is_eu",
    "position", "main_position", "foot", "height",
    "current_club_id", "current_club_name",
    "joined", "contract_expires",
]
# se crea la tabla base con datos personales, depportivos y contractuales solo de los jovenes, sin embarog se conservan los que no tengan lesiones, tranferencias o stats, el join se hace por player_id
base = young[[c for c in keep_profile if c in young.columns]].copy()

# Merges encadenados, se agregan las medidas calculadas previamente
base = base.merge(mv_latest[["player_id", "latest_value", "latest_date"]],  on="player_id", how="left")
base = base.merge(mv_agg,                                                     on="player_id", how="left")
base = base.merge(career_stats,                                               on="player_id", how="left")
base = base.merge(inj_agg,                                                    on="player_id", how="left")
base = base.merge(nat_agg,                                                    on="player_id", how="left")
base = base.merge(transfer_agg,                                               on="player_id", how="left")

# Info de club/liga
team_slim = (
    team_det[["club_id", "country_name", "competition_name"]]
    .drop_duplicates("club_id")
    .rename(columns={"club_id": "current_club_id", "country_name": "club_country",
                     "competition_name": "club_league"})
)
base = base.merge(team_slim, on="current_club_id", how="left")

# Rellenar nulos de contadores con 0 (jugadores sin lesiones/internacionales/transferencias), evitamo errores en el modelo y hacemos que la data sea coherente
fill_zero = [
    "n_injuries", "total_days_missed", "total_games_missed",
    "intl_matches", "intl_goals", "n_nat_teams", "is_current_national",
    "n_transfers", "max_fee", "total_fee",
]
base[fill_zero] = base[[c for c in fill_zero if c in base.columns]].fillna(0)

# Variable objetivo en escala log para reducir el sesgo por valores grandes, ademas mejora el rendimiento del modelo
base["log_market_value"] = np.log1p(base["latest_value"])

# Filtro final: solo jugadores con al menos 1 temporada de stats
final = base[base["n_seasons_active"].notna() & (base["n_seasons_active"] > 0)].copy()

print(f"\n  DATASET FINAL: {len(final):,} jugadores × {final.shape[1]} variables")
print(f"  Con valor de mercado (target): {final['latest_value'].notna().sum():,} ({final['latest_value'].notna().mean()*100:.1f}%)")

# Guardar
csv_path = OUTPUT_DIR / "scouting_dataset_v1.csv"
final.to_csv(csv_path, index=False)
print(f"\n  ✓ Dataset guardado: {csv_path}")

# Muestra de variables disponibles
print(f"\n  Columnas del dataset:\n  {list(final.columns)}")


══════════════════════════════════════════════════════════════
  4. CONSTRUCCIÓN DEL DATASET MAESTRO
══════════════════════════════════════════════════════════════
  Base — jugadores ≤26 años con club activo: 33,258

  DATASET FINAL: 31,405 jugadores × 48 variables
  Con valor de mercado (target): 17,035 (54.2%)

  ✓ Dataset guardado: ..\outputs\paso1\scouting_dataset_v1.csv

  Columnas del dataset:
  ['player_id', 'player_name', 'date_of_birth', 'age', 'country_of_birth', 'citizenship', 'is_eu', 'position', 'main_position', 'foot', 'height', 'current_club_id', 'current_club_name', 'joined', 'contract_expires', 'latest_value', 'latest_date', 'value_max', 'value_mean', 'n_valuations', 'value_first', 'value_last', 'date_first', 'date_last', 'value_growth_abs', 'value_growth_pct', 'total_goals', 'total_assists', 'total_minutes', 'total_yellow', 'total_red', 'n_competitions', 'n_seasons_active', 'goal_contrib', 'mins_per_goal', 'n_injuries', 'total_days_missed', 'total_games_missed', 'int

### Hallazgos del dataset maestro

**Dataset final: 31,405 jugadores × 48 variables**

El merge funcionó correctamente en todas las fuentes. Los puntos críticos:

- **Target (`latest_value`):** solo el 54.2% de los jugadores tiene valor de mercado
  registrado (17,035 jugadores). Esto es esperado — Transfermarkt solo valoriza
  jugadores que tienen cierta exposición mediática o que juegan en ligas monitoreadas.
  Para el modelo supervisado se usarán solo estos 17,035.

- **Valor histórico (`value_max`, `value_growth_pct`):** 70.9% de cobertura.
  Los jugadores sin historial suelen ser muy jóvenes o de ligas poco seguidas.

- **El dataset resultante cubre un espectro muy amplio:** desde un juvenil de 14 años
  en una academia hasta jugadores de 26 años en clubes de primer nivel como
  Real Madrid o Manchester City.

>  **Nota importante:** el 45.8% de jugadores sin valor de mercado registrado
> no se descarta — en el modelo se evaluará si se puede usar aprendizaje
> semi-supervisado o si simplemente se entrenan con el subconjunto con target disponible.

## 5. Visualizaciones EDA

Se generan 10 visualizaciones que cubren:
1. Calidad de datos (nulos)
2. Distribución del valor de mercado (original y log-transformada)
3. Curva valor de mercado por edad
4. Análisis por posición
5. Correlaciones de features con el target
6. Top 25 promesas actuales
7. Evolución temporal del valor (top 8)
8. Impacto de lesiones en el valor
9. Heatmap de correlaciones entre features
10. Distribución geográfica

In [12]:
# 5. VISUALIZACIONES EDA (10 plots)
# ──────────────────────────────────────────────────────────────

section("5. VISUALIZACIONES EDA")

df_v = final[final["latest_value"].notna() & (final["latest_value"] > 0)].copy()

# ── Plot 1: Calidad de datos (% nulos)
null_pct = (final.isnull().mean() * 100).round(1).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(12, max(5, len(null_pct) * 0.38)))
bar_colors = [
    PALETTE["danger"]    if v > 50 else
    PALETTE["accent"]    if v > 20 else
    PALETTE["secondary"] if v > 5  else
    PALETTE["primary"]
    for v in null_pct.values
]
bars = ax.barh(null_pct.index, null_pct.values, color=bar_colors, height=0.65)
ax.axvline(20, color=PALETTE["accent"], ls="--", lw=1.2, alpha=0.8, label="20% — revisar imputación")
ax.axvline(50, color=PALETTE["danger"], ls="--", lw=1.2, alpha=0.8, label="50% — considerar descarte")
ax.set_xlabel("% de valores nulos")
ax.set_title("Calidad de datos — % nulos por variable\n(dataset scouting jóvenes promesas ≤26 años)",
             fontweight="bold", pad=12)
ax.legend(fontsize=9)
for bar, val in zip(bars, null_pct.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{val:.0f}%", va="center", fontsize=8)
fig.tight_layout()
save_fig(fig, "01_calidad_nulos")

# ── Plot 2: Distribución del valor de mercado
data_mv = df_v["latest_value"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Variable objetivo: Valor de mercado (jugadores ≤26 años)", fontsize=13, fontweight="bold")

axes[0].hist(data_mv / 1e6, bins=70, color=PALETTE["primary"], edgecolor="white", lw=0.3)
axes[0].set_xscale("log")
axes[0].set_xlabel("Valor de mercado (M€) — escala log")
axes[0].set_ylabel("Jugadores")
axes[0].set_title("Distribución original")
p50 = data_mv.median() / 1e6
p75 = data_mv.quantile(0.75) / 1e6
axes[0].axvline(p50, color=PALETTE["accent"], lw=1.8, ls="--", label=f"Mediana: {p50:.2f}M€")
axes[0].axvline(p75, color=PALETTE["danger"], lw=1.5, ls=":",  label=f"P75: {p75:.1f}M€")
axes[0].legend(fontsize=9)

log_vals = np.log1p(data_mv)
axes[1].hist(log_vals, bins=60, color=PALETTE["secondary"], edgecolor="white", lw=0.3)
axes[1].set_xlabel("log(1 + valor €)")
axes[1].set_ylabel("Jugadores")
axes[1].set_title("Log-transformada (target para el modelo)")
axes[1].axvline(log_vals.mean(), color=PALETTE["accent"], lw=1.8, ls="--",
                label=f"Media log: {log_vals.mean():.2f}")
axes[1].legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "02_distribucion_valor_mercado")

# ── Plot 3: Valor de mercado por edad
age_g = df_v.groupby(df_v["age"].round(0).astype(int))["latest_value"].agg(["median","mean","count"])
age_g = age_g[age_g["count"] >= 15]

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(age_g.index, age_g["median"]/1e6, alpha=0.18, color=PALETTE["primary"])
ax.plot(age_g.index, age_g["median"]/1e6, color=PALETTE["primary"], lw=2.5, label="Mediana")
ax.plot(age_g.index, age_g["mean"]/1e6,   color=PALETTE["secondary"], lw=1.8, ls="--", label="Media")
ax2 = ax.twinx()
ax2.bar(age_g.index, age_g["count"], alpha=0.12, color=PALETTE["neutral"], width=0.7)
ax2.set_ylabel("Nº jugadores", color=PALETTE["neutral"], fontsize=9)
ax2.tick_params(axis="y", colors=PALETTE["neutral"])
pico = age_g["median"].idxmax()
pico_v = age_g.loc[pico, "median"] / 1e6
ax.annotate(f"Pico: {pico} años\n{pico_v:.1f}M€",
    xy=(pico, pico_v), xytext=(pico+0.8, pico_v * 0.8),
    arrowprops=dict(arrowstyle="->", color=PALETTE["neutral"], lw=1.2),
    fontsize=9, color=PALETTE["neutral"])
ax.set_xlabel("Edad")
ax.set_ylabel("Valor de mercado (M€)")
ax.set_title("Curva de valor de mercado por edad — promesas ≤26 años", fontweight="bold")
ax.legend(loc="upper left")
fig.tight_layout()
save_fig(fig, "03_valor_por_edad")

# ── Plot 4: Valor por posición
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Análisis por posición (main_position)", fontsize=13, fontweight="bold")

pos_cnt = final["main_position"].value_counts()
clrs = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["danger"]]
axes[0].bar(pos_cnt.index, pos_cnt.values, color=clrs[:len(pos_cnt)], width=0.6)
axes[0].set_ylabel("Jugadores")
axes[0].set_title("Cantidad por posición")
for i, (pos, cnt) in enumerate(pos_cnt.items()):
    axes[0].text(i, cnt + 40, f"{cnt:,}", ha="center", fontsize=10, fontweight="bold")

pos_val = df_v.groupby("main_position")["latest_value"].agg(["median","mean"]) / 1e6
pos_val = pos_val.sort_values("median", ascending=False)
x = np.arange(len(pos_val)); w = 0.35
axes[1].bar(x - w/2, pos_val["median"], w, label="Mediana", color=PALETTE["primary"])
axes[1].bar(x + w/2, pos_val["mean"],   w, label="Media",   color=PALETTE["secondary"], alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(pos_val.index)
axes[1].set_ylabel("Valor (M€)")
axes[1].set_title("Valor de mercado por posición")
axes[1].legend()
fig.tight_layout()
save_fig(fig, "04_valor_por_posicion")

# ── Plot 5: Correlaciones con el target
num_cols = [
    "age","height","total_goals","total_assists","total_minutes",
    "total_yellow","total_red","n_competitions","n_seasons_active",
    "mins_per_goal","goal_contrib","n_injuries","total_days_missed",
    "intl_matches","intl_goals","is_current_national",
    "n_transfers","max_fee","value_max","value_growth_pct","n_valuations",
]
num_cols = [c for c in num_cols if c in final.columns]
corr = final[num_cols + ["log_market_value"]].corr()["log_market_value"].drop("log_market_value").sort_values()

fig, ax = plt.subplots(figsize=(10, max(5, len(corr) * 0.45)))
colors_corr = [PALETTE["primary"] if v > 0 else PALETTE["danger"] for v in corr.values]
ax.barh(corr.index, corr.values, color=colors_corr, height=0.65)
ax.axvline(0,    color=PALETTE["dark"],    lw=0.8)
ax.axvline(0.3,  color=PALETTE["primary"], lw=0.8, ls=":", alpha=0.6, label="±0.30 (correlación moderada)")
ax.axvline(-0.3, color=PALETTE["primary"], lw=0.8, ls=":", alpha=0.6)
ax.set_xlabel("Correlación de Pearson con log(valor de mercado)")
ax.set_title("Features vs target — correlaciones con log(valor de mercado)", fontweight="bold")
ax.legend(fontsize=9); ax.set_xlim(-1, 1)
for bar, val in zip(ax.patches, corr.values):
    xpos = bar.get_width() + 0.01 if val >= 0 else bar.get_width() - 0.01
    ha   = "left" if val >= 0 else "right"
    ax.text(xpos, bar.get_y() + bar.get_height()/2, f"{val:.2f}",
            va="center", ha=ha, fontsize=8)
fig.tight_layout()
save_fig(fig, "05_correlaciones_target")

# ── Plot 6: Top 25 promesas
top25 = (
    df_v[["player_name","age","main_position","latest_value","country_of_birth","current_club_name"]]
    .sort_values("latest_value", ascending=False)
    .head(25)
)
fig, ax = plt.subplots(figsize=(12, 9))
colors_pos = {
    "Attack": PALETTE["danger"], "Midfield": PALETTE["secondary"],
    "Defender": PALETTE["primary"], "Goalkeeper": PALETTE["accent"],
}
bar_colors = [colors_pos.get(p, PALETTE["neutral"]) for p in top25["main_position"]]
ax.barh(range(len(top25)), top25["latest_value"] / 1e6, color=bar_colors, height=0.72)
ax.set_yticks(range(len(top25)))
ax.set_yticklabels([
    f"{r['player_name'].split('(')[0].strip()} · {int(r['age'])}a · {r['main_position']}"
    for _, r in top25.iterrows()
], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Valor de mercado (M€)")
ax.set_title("Top 25 jóvenes promesas ≤26 años — valor de mercado actual", fontweight="bold", pad=12)
from matplotlib.patches import Patch
legend_el = [Patch(color=c, label=l) for l, c in colors_pos.items()]
ax.legend(handles=legend_el, loc="lower right", fontsize=9)
for i, (_, row) in enumerate(top25.iterrows()):
    ax.text(row["latest_value"]/1e6 + 0.5, i, f"{row['latest_value']/1e6:.0f}M€",
            va="center", fontsize=8.5, color=PALETTE["dark"])
fig.tight_layout()
save_fig(fig, "06_top25_promesas")

# ── Plot 7: Evolución temporal top 8
top8_pids = top25.head(8)["player_name"].tolist()
top8_ids  = final[final["player_name"].isin(top8_pids)]["player_id"].tolist()
name_map  = final.set_index("player_id")["player_name"].str.split("(").str[0].str.strip().to_dict()

mv_hist_raw = pd.read_csv(DATA_DIR / "player_market_value.csv")
mv_hist_raw["date_unix"] = pd.to_datetime(mv_hist_raw["date_unix"], errors="coerce")
hist_top = mv_hist_raw[mv_hist_raw["player_id"].isin(top8_ids) & (mv_hist_raw["value"] > 0)].copy()

fig, ax = plt.subplots(figsize=(14, 6))
palette_evo = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"],
               PALETTE["danger"], "#4A90D9", "#7ED321", "#9013FE", "#F5A623"]
for i, (pid, grp) in enumerate(hist_top.groupby("player_id")):
    grp = grp.sort_values("date_unix")
    ax.plot(grp["date_unix"], grp["value"]/1e6,
            label=name_map.get(pid, str(pid)),
            color=palette_evo[i % len(palette_evo)], lw=2, marker="o", ms=2.5)
ax.set_xlabel("Fecha"); ax.set_ylabel("Valor de mercado (M€)")
ax.set_title("Evolución del valor de mercado — Top 8 promesas", fontweight="bold")
ax.legend(loc="upper left", fontsize=9, ncol=2)
fig.tight_layout()
save_fig(fig, "07_evolucion_temporal")

# ── Plot 8: Lesiones vs valor de mercado
inj_df = final[
    final["latest_value"].notna() & (final["latest_value"] > 0) & (final["n_injuries"] > 0)
].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Impacto de lesiones en el valor de mercado", fontsize=13, fontweight="bold")
for i, (xcol, xlabel) in enumerate([("n_injuries","Nº lesiones"), ("total_days_missed","Días de baja total")]):
    sc = axes[i].scatter(inj_df[xcol], inj_df["latest_value"]/1e6,
                         c=inj_df["age"], cmap="viridis", alpha=0.4, s=20)
    axes[i].set_xlabel(xlabel); axes[i].set_ylabel("Valor de mercado (M€)")
    axes[i].set_yscale("log")
    axes[i].set_title(f"{xlabel} vs valor de mercado")
plt.colorbar(sc, ax=axes[1], label="Edad")
fig.tight_layout()
save_fig(fig, "08_lesiones_vs_valor")

# ── Plot 9: Heatmap de correlaciones
heat_vars = [
    "log_market_value","age","total_goals","total_assists","total_minutes",
    "n_competitions","n_seasons_active","goal_contrib","n_injuries",
    "total_days_missed","intl_matches","intl_goals","is_current_national",
    "n_transfers","max_fee","value_max","value_growth_pct",
]
heat_vars = [c for c in heat_vars if c in final.columns]
corr_mat  = final[heat_vars].corr()
fig, ax   = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, linewidths=0.3, annot_kws={"size": 8}, ax=ax, vmin=-1, vmax=1)
ax.set_title("Mapa de correlaciones — features del modelo de scouting", fontweight="bold", pad=12)
fig.tight_layout()
save_fig(fig, "09_heatmap_correlaciones")

# ── Plot 10: Distribución geográfica
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Geografía de las promesas (≤26 años)", fontsize=13, fontweight="bold")

top_c = final["country_of_birth"].value_counts().head(15)
axes[0].bar(range(len(top_c)), top_c.values, color=PALETTE["secondary"], width=0.7)
axes[0].set_xticks(range(len(top_c)))
axes[0].set_xticklabels(top_c.index, rotation=40, ha="right", fontsize=9)
axes[0].set_title("Top 15 países por número de promesas")
axes[0].set_ylabel("Jugadores")

country_val = (
    df_v.groupby("country_of_birth")["latest_value"]
    .agg(["median","count"])
)
country_val = country_val[country_val["count"] >= 10].sort_values("median", ascending=False).head(15)
axes[1].bar(range(len(country_val)), country_val["median"]/1e6, color=PALETTE["primary"], width=0.7)
axes[1].set_xticks(range(len(country_val)))
axes[1].set_xticklabels(country_val.index, rotation=40, ha="right", fontsize=9)
axes[1].set_title("Valor mediano por país (mín. 10 jugadores)")
axes[1].set_ylabel("Valor mediano (M€)")
fig.tight_layout()
save_fig(fig, "10_distribucion_geografica")


══════════════════════════════════════════════════════════════
  5. VISUALIZACIONES EDA
══════════════════════════════════════════════════════════════
  ✓ 01_calidad_nulos.png
  ✓ 02_distribucion_valor_mercado.png
  ✓ 03_valor_por_edad.png
  ✓ 04_valor_por_posicion.png
  ✓ 05_correlaciones_target.png
  ✓ 06_top25_promesas.png
  ✓ 07_evolucion_temporal.png
  ✓ 08_lesiones_vs_valor.png
  ✓ 09_heatmap_correlaciones.png
  ✓ 10_distribucion_geografica.png


### Hallazgos de las visualizaciones

**Plot 2 — Distribución del valor de mercado:**  
La distribución original es extremadamente sesgada a la derecha (pocos jugadores 
con valores muy altos concentran la mayor parte del rango). La log-transformación 
produce una distribución casi normal — esto confirma que `log_market_value` es 
la variable correcta como target para el modelo.

**Plot 3 — Curva valor por edad:**  
El pico de valor mediano ocurre alrededor de los 23-24 años. Esto tiene sentido 
económico: a esa edad los jugadores combinan potencial futuro + rendimiento 
demostrado, que es exactamente lo que el mercado paga. Después de los 24 el valor 
empieza a estabilizarse o caer levemente, con excepciones notorias (Mbappé, Haaland).

**Plot 5 — Correlaciones con el target:**  
Las variables más correlacionadas con `log_market_value` son:
- `value_max` (correlación muy alta — el máximo histórico es un predictor fuerte 
  del valor actual, pero hay que tener cuidado con data leakage si el máximo 
  coincide con el valor actual)
- `intl_matches` — los jugadores internacionales valen significativamente más
- `max_fee` — el fee máximo pagado es una señal de mercado externa muy potente
- `goal_contrib` y `total_minutes` — rendimiento sostenido correlaciona con valor

Variables con correlación negativa o casi nula: `total_yellow`, `total_red`, 
`n_injuries` — curiosamente las lesiones no muestran correlación negativa fuerte, 
posiblemente porque los jugadores más valiosos también son los más monitoreados 
en términos de salud.

**Plot 6 — Top 25 promesas:**  
Lamine Yamal (17 años, 200M€) y Mbappé/Haaland/Bellingham (180M€) encabezan 
la lista. Destaca el dominio del mediocampo y el ataque — solo 2 defensas 
(Bastoni y Saliba) entran en el top 25, lo que refleja el sesgo histórico del 
mercado hacia posiciones ofensivas.

**Plot 9 — Heatmap:**  
Se confirma multicolinealidad entre `value_max`, `value_mean` y `latest_value` — 
en el Paso 2 habrá que decidir cuál conservar o aplicar PCA/selección de features 
para evitar redundancia.

En la carpeta ../outputs/paso1 se encuentra cada uno de los graficos generados

## 6. Calidad de datos

Se analiza el porcentaje de nulos por variable en el dataset final.
Esto determina la estrategia de imputación que se usará en el Paso 2.

**Criterios:**
- ✓ ≥ 80% completo → listo para modelado directo
- ~ 50–79% → imputar con mediana por posición o KNN
- ✗ < 50% → evaluar descarte o ingeniería alternativa

In [14]:
# 6. REPORTE DE FEATURES
# ──────────────────────────────────────────────────────────────
section("6. REPORTE DE FEATURES PARA EL MODELO")

feature_groups = {
    "PERFIL (demográfico)": [
        "age", "height", "foot", "country_of_birth", "citizenship", "is_eu", "main_position"
    ],
    "RENDIMIENTO (carrera completa)": [
        "total_goals", "total_assists", "total_minutes", "goal_contrib",
        "mins_per_goal", "n_seasons_active", "n_competitions",
        "total_yellow", "total_red"
    ],
    "LESIONES (factor de riesgo)": [
        "n_injuries", "total_days_missed", "total_games_missed"
    ],
    "SELECCIÓN NACIONAL (presión internacional)": [
        "intl_matches", "intl_goals", "is_current_national"
    ],
    "VALOR HISTÓRICO (trayectoria económica)": [
        "value_max", "value_mean", "value_growth_abs", "value_growth_pct", "n_valuations"
    ],
    "TRANSFERENCIAS (señal de mercado)": [
        "n_transfers", "max_fee", "total_fee"
    ],
    "CONTEXTO (club/liga)": [
        "club_country", "club_league"
    ],
    "TARGET (variable objetivo)": [
        "latest_value", "log_market_value"
    ],
}

print()
for grupo, cols in feature_groups.items():
    print(f"  ── {grupo}")
    for c in cols:
        if c in final.columns:
            pct = final[c].notna().mean() * 100
            status = "✓" if pct >= 80 else "~" if pct >= 50 else "✗"
            print(f"     {status}  {c:<38} {pct:5.1f}% completo")
        else:
            print(f"     –  {c:<38} no disponible en dataset")
    print()


══════════════════════════════════════════════════════════════
  6. REPORTE DE FEATURES PARA EL MODELO
══════════════════════════════════════════════════════════════

  ── PERFIL (demográfico)
     ✓  age                                    100.0% completo
     ✓  height                                 100.0% completo
     ~  foot                                    76.7% completo
     ~  country_of_birth                        75.2% completo
     ✓  citizenship                            100.0% completo
     ✓  is_eu                                  100.0% completo
     ✓  main_position                          100.0% completo

  ── RENDIMIENTO (carrera completa)
     ✓  total_goals                            100.0% completo
     ✓  total_assists                          100.0% completo
     ✓  total_minutes                          100.0% completo
     ✓  goal_contrib                           100.0% completo
     ~  mins_per_goal                           71.8% completo
     ✓  n_sea

### Hallazgos de calidad

**Variables con cobertura perfecta (✓):**  
`age`, `height`, `citizenship`, `is_eu`, `main_position`, `total_goals`,
`total_assists`, `total_minutes`, `n_seasons_active`, `n_injuries`,
`intl_matches`, `n_transfers` — todas al 100%.  
Estas son el núcleo sólido del modelo.

**Variables a imputar (~):**  
- `foot` (76.7%) — se puede imputar con la moda por posición (los delanteros 
  derechos son mayoría) o dejarlo como categoría "unknown".
- `country_of_birth` (75.2%) — se puede derivar de `citizenship` en la mayoría de casos.
- `mins_per_goal` (71.8%) — nulo cuando el jugador no ha marcado goles (porteros, 
  defensas). Se puede imputar con un valor alto (ej. 9999) o crear una feature 
  binaria `es_goleador`.
- Variables del historial de valor (70.9%) — imputar con mediana por posición y liga.

**Variable objetivo (~):**  
`latest_value` al 54.2% — suficiente para un modelo supervisado robusto con 17,035 casos.
No se considera problemático dado el dominio.

## 7. Conclusiones del Paso 1

- Dataset de **31,405 jugadores promesas** (≤26 años) con 48 variables integradas
  de 8 fuentes distintas.
- **17,035 casos con variable objetivo** (`latest_value`) disponible para entrenamiento supervisado.
- Identificación clara de features sólidas, features a imputar y posibles redundancias.
- Variable objetivo confirmada: `log_market_value` (distribución aproximadamente normal).

In [15]:
section("RESUMEN FINAL")
print(f"""
  Dataset curado:   {OUTPUT_DIR}/scouting_dataset_v1.csv
  Visualizaciones:  {OUTPUT_DIR}/ (10 plots PNG)

  Estadísticas del dataset final:
    Jugadores totales:              {len(final):,}
    Con valor de mercado (target):  {final['latest_value'].notna().sum():,} ({final['latest_value'].notna().mean()*100:.1f}%)
    Variables (features + target):  {final.shape[1]}
    Rango de edad:                  {final['age'].min():.0f} – {final['age'].max():.0f} años
    Valor máximo en el subset:      {final['latest_value'].max()/1e6:.0f}M€
    Posiciones:                     {final['main_position'].nunique()} categorías
    Países de origen:               {final['country_of_birth'].nunique()} países

  Leyenda calidad features:
    ✓  ≥80% completo — listo para modelado
    ~  50–79% — imputar con mediana/moda antes de entrenar
    ✗  <50%   — considerar descarte o ingeniería alternativa

  Próximo paso → paso2_modelo_predictivo.py
    · Feature engineering (encoding posiciones, ligas, pierna)
    · Imputación de nulos (KNN / mediana por posición)
    · Entrenamiento LightGBM + XGBoost
    · Tracking de experimentos con MLflow
""")


══════════════════════════════════════════════════════════════
  RESUMEN FINAL
══════════════════════════════════════════════════════════════

  Dataset curado:   ..\outputs\paso1/scouting_dataset_v1.csv
  Visualizaciones:  ..\outputs\paso1/ (10 plots PNG)

  Estadísticas del dataset final:
    Jugadores totales:              31,405
    Con valor de mercado (target):  17,035 (54.2%)
    Variables (features + target):  48
    Rango de edad:                  15 – 26 años
    Valor máximo en el subset:      200M€
    Posiciones:                     4 categorías
    Países de origen:               181 países

  Leyenda calidad features:
    ✓  ≥80% completo — listo para modelado
    ~  50–79% — imputar con mediana/moda antes de entrenar
    ✗  <50%   — considerar descarte o ingeniería alternativa

  Próximo paso → paso2_modelo_predictivo.py
    · Feature engineering (encoding posiciones, ligas, pierna)
    · Imputación de nulos (KNN / mediana por posición)
    · Entrenamiento LightGBM + X